# 01 — Official Source Collection and Collection Audit

This notebook documents the collection strategy used for the Dutch Politically Exposed Person (PEP) benchmark.

The benchmark was constructed through a combination of:

1. an initial automated extraction and manual screening stage, retained as a frozen curated baseline;
2. source-specific extraction for structured official sources;
3. browser-based extraction for dynamic official pages;
4. manually verified official-source additions where automated extraction was unreliable or blocked;
5. manually coded Category C party-board records from official party websites.

The notebook creates an auditable source registry and collection log. It does not claim that one generic scraper can reliably extract every PEP category.

The reviewed baseline benchmark is stored in:

`data/input/curated_intermediates/benchmark_v1_curated.csv`

The final benchmark is constructed in:

`02_clean_build_benchmark.ipynb`

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import requests
import re
import unicodedata

# -------------------------------------------------------------------
# Project folders
# -------------------------------------------------------------------
# This notebook is stored in Code_clean/notebooks.
# The project folder is therefore one level above the current folder.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
CURATED_INTERMEDIATES_DIR = INPUT_DIR / "curated_intermediates"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
RAW_HTML_DIR = PROJECT_DIR / "data" / "raw_html"
RAW_TEXT_DIR = PROJECT_DIR / "data" / "raw_text"

# Create folders that will contain generated outputs.
for folder in [OUTPUT_DIR, RAW_HTML_DIR, RAW_TEXT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Input file paths
# -------------------------------------------------------------------

PEP_SOURCE_MAPPING_PATH = INPUT_DIR / "PEP_source_mapping.csv"
TAXONOMY_PATH = INPUT_DIR / "taxonomy_v2_eu_aligned.xlsx"
MANUAL_ADDITIONS_PATH = INPUT_DIR / "manual_main_v2_additions.xlsx"
CATEGORY_C_SOURCES_PATH = INPUT_DIR / "category_c_party_sources.csv"
CATEGORY_C_MANUAL_PATH = INPUT_DIR / "category_c_party_board_manual_v3.csv"
CURATED_V1_PATH = (
    CURATED_INTERMEDIATES_DIR / "benchmark_v1_curated.csv"
)

print("Project folder:", PROJECT_DIR)

print("\nKey inputs:")
print("PEP source mapping exists:", PEP_SOURCE_MAPPING_PATH.exists())
print("Taxonomy workbook exists:", TAXONOMY_PATH.exists())
print("Manual v2 additions exist:", MANUAL_ADDITIONS_PATH.exists())
print("Category C sources exist:", CATEGORY_C_SOURCES_PATH.exists())
print("Manual Category C records exist:", CATEGORY_C_MANUAL_PATH.exists())
print("Curated v1 benchmark exists:", CURATED_V1_PATH.exists())

Project folder: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean

Key inputs:
PEP source mapping exists: True
Taxonomy workbook exists: True
Manual v2 additions exist: True
Category C sources exist: True
Manual Category C records exist: True
Curated v1 benchmark exists: True


## 1. Load collection protocol inputs

This section loads the files that document how sources were selected and how each benchmark component was collected.

The source mapping provides the original official source list. The taxonomy workbook provides the EU-aligned category mapping, benchmark scope decisions, and documented extraction strategies.

The curated benchmark v1 and manually reviewed additions are retained as versioned research inputs because they contain review decisions that cannot necessarily be reproduced from changing live websites.

In [2]:
# -------------------------------------------------------------------
# Helper function: load CSV files saved with common Windows encodings
# -------------------------------------------------------------------
# Some CSV files saved through Excel use cp1252 encoding rather than
# UTF-8. This helper tries several common encodings in a safe order.

def read_csv_with_fallback_encodings(file_path):
    """
    Load a CSV file using common encodings.

    Returns:
    - dataframe: the loaded CSV data
    - used_encoding: the encoding that successfully loaded the file
    """
    possible_encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in possible_encodings:
        try:
            dataframe = pd.read_csv(
                file_path,
                encoding=encoding
            )

            print(f"Loaded {file_path.name} using encoding: {encoding}")

            return dataframe, encoding

        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not load {file_path.name}. "
        f"Last encoding error: {last_error}"
    )

In [3]:
# -------------------------------------------------------------------
# Load source mapping, curated benchmark, and manual source inputs
# -------------------------------------------------------------------

pep_source_mapping_df, pep_source_mapping_encoding = (
    read_csv_with_fallback_encodings(PEP_SOURCE_MAPPING_PATH)
)

category_c_sources_df, category_c_sources_encoding = (
    read_csv_with_fallback_encodings(CATEGORY_C_SOURCES_PATH)
)

category_c_manual_df, category_c_manual_encoding = (
    read_csv_with_fallback_encodings(CATEGORY_C_MANUAL_PATH)
)

curated_v1_df, curated_v1_encoding = (
    read_csv_with_fallback_encodings(CURATED_V1_PATH)
)

manual_additions_sheet_names = pd.ExcelFile(
    MANUAL_ADDITIONS_PATH
).sheet_names

taxonomy_sheet_names = pd.ExcelFile(
    TAXONOMY_PATH
).sheet_names

print("\nInput row counts:")
print("PEP source mapping:", len(pep_source_mapping_df))
print("Category C official-source list:", len(category_c_sources_df))
print("Manual Category C records:", len(category_c_manual_df))
print("Curated benchmark v1:", len(curated_v1_df))

print("\nManual additions workbook sheets:")
print(manual_additions_sheet_names)

print("\nTaxonomy workbook sheets:")
print(taxonomy_sheet_names)

Loaded PEP_source_mapping.csv using encoding: cp1252
Loaded category_c_party_sources.csv using encoding: utf-8
Loaded category_c_party_board_manual_v3.csv using encoding: utf-8
Loaded benchmark_v1_curated.csv using encoding: utf-8

Input row counts:
PEP source mapping: 72
Category C official-source list: 15
Manual Category C records: 79
Curated benchmark v1: 286

Manual additions workbook sheets:
['cbb', 'crvb', 'senior_military']

Taxonomy workbook sheets:
['canonical_eu_categories', 'country_function_mapping', 'nl_source_mapping_detail', 'extraction_strategy_registry', 'scope_decisions', 'coding_codebook', 'eu_country_function_coding']


## 2. Load taxonomy mapping and collection-strategy documentation

The taxonomy workbook contains the mapping from Dutch PEP functions to EU-aligned categories.

The `nl_source_mapping_detail` sheet identifies the Dutch functions and official sources considered for the benchmark. The `extraction_strategy_registry` sheet records which collection approach was considered appropriate for each source type.

These sheets are used to create an auditable collection-strategy registry.

In [4]:
# -------------------------------------------------------------------
# Load the taxonomy sheets needed for collection documentation
# -------------------------------------------------------------------

nl_source_mapping_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name="nl_source_mapping_detail"
)

strategy_registry_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name="extraction_strategy_registry"
)

scope_decisions_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name="scope_decisions"
)

# Standardise column names so later code is easier to read and reuse.
def clean_column_names(dataframe):
    """
    Convert column names to a consistent snake_case format.
    """
    dataframe = dataframe.copy()

    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return dataframe


nl_source_mapping_df = clean_column_names(nl_source_mapping_df)
strategy_registry_df = clean_column_names(strategy_registry_df)
scope_decisions_df = clean_column_names(scope_decisions_df)

print("Dutch source mapping rows:", len(nl_source_mapping_df))
print("Extraction strategy rows:", len(strategy_registry_df))
print("Scope-decision rows:", len(scope_decisions_df))

print("\nDutch source mapping columns:")
print(nl_source_mapping_df.columns.tolist())

print("\nExtraction strategy columns:")
print(strategy_registry_df.columns.tolist())

print("\nScope-decision columns:")
print(scope_decisions_df.columns.tolist())

Dutch source mapping rows: 16
Extraction strategy rows: 5
Scope-decision rows: 2

Dutch source mapping columns:
['country_code', 'canonical_category', 'local_taxonomy_category', 'role_nl', 'role_eng', 'source_url', 'source_type', 'extraction_strategy', 'included_in_main_benchmark', 'operationalisation_status', 'notes']

Extraction strategy columns:
['extraction_strategy', 'description', 'suitable_for', 'requires_manual_validation', 'example_category']

Scope-decision columns:
['decision_area', 'decision', 'reasoning', 'implication']


In [5]:
# -------------------------------------------------------------------
# Inspect documented Dutch benchmark sources and collection strategies
# -------------------------------------------------------------------

display(
    nl_source_mapping_df.head(15)
)

display(
    strategy_registry_df.head(15)
)

display(
    scope_decisions_df.head(15)
)

,country_code,canonical_category,local_taxonomy_category,role_nl,role_eng,source_url,source_type,extraction_strategy,included_in_main_benchmark,operationalisation_status,notes
0,NL,executive_leadership,head_of_state,Koning,King,https://www.koninklijkhuis.nl/leden-koninklijk...,official_html_page,manual_official_source_addition,Yes,Direct,Small official category; manually verified
1,NL,executive_leadership,national_executive,Minister-president; ministers; staatssecretari...,Prime Minister; ministers; state secretaries,https://www.rijksoverheid.nl/regering/bewindsp...,official_html_page,source_specific_html_selector,Yes,Direct,Already included in benchmark v1
2,NL,legislative_body,senate_members,Leden van de Eerste Kamer der Staten-Generaal,Members of the Senate,https://www.eerstekamer.nl/alle_leden,official_html_page,source_specific_html_selector,Yes,Direct,Extracted using div.naam selector
3,NL,legislative_body,house_members,Leden van de Tweede Kamer der Staten-Generaal,Members of the House of Representatives,https://www.tweedekamer.nl/kamerleden_en_commi...,official_html_page,source_specific_html_selector,Yes,Direct,Already included in benchmark v1
4,NL,political_party_governance,registered_party_board,Leden van het bestuur van een politieke groepe...,Members of registered political party boards,manual selected party board pages,official_html_page,manual_official_source_addition,No,Direct,Included in benchmark v3 as an extended compon...
5,NL,high_judiciary,high_judiciary_rvs,Leden en staatsraden benoemd in de Afdeling be...,Members and State Councillors of the Administr...,https://www.raadvanstate.nl/overrvs/organisati...,official_html_page,source_specific_html_selector,Yes,Direct,Already included in benchmark v1
6,NL,high_judiciary,high_judiciary_hr,President; vice-presidenten; raadsheren; raads...,President; vice-presidents; judges; extraordin...,https://www.hogeraad.nl/hoge-raad/kamers/,official_html_page,source_specific_html_selector,Yes,Direct,Already included in benchmark v1
7,NL,high_judiciary,high_judiciary_cbb,Leden met rechtspraak belast van het College v...,Members of the Trade and Industry Appeals Trib...,https://www.rechtspraak.nl/organisatie-en-cont...,official_html_page,manual_official_source_addition,Yes,Direct,Automated access returned HTTP 403; manual off...
8,NL,high_judiciary,high_judiciary_crvb,Leden met rechtspraak belast van de Centrale R...,Members of the Central Appeals Tribunal charge...,https://www.rechtspraak.nl/organisatie-en-cont...,official_html_page,manual_official_source_addition,Yes,Direct,Automated access returned HTTP 403; manual off...
9,NL,audit_and_central_bank,court_of_audit,Leden in gewone en buitengewone dienst van de ...,Members of the Netherlands Court of Audit,https://www.rekenkamer.nl/over-de-algemene-rek...,official_html_page,manual_official_source_addition,Yes,Direct,Small official category; manually verified


,extraction_strategy,description,suitable_for,requires_manual_validation,example_category
0,source_specific_html_selector,Uses source-specific HTML selectors when an of...,Structured official HTML pages,Yes,Eerste Kamer; Tweede Kamer; Raad van State
1,manual_official_source_addition,Manual addition based on a small official sour...,Small official categories with few names,Yes,Head of state; Algemene Rekenkamer; DNB; CBB; ...
2,manual_download_parse,Manual browser download of source HTML followe...,Official pages that block automated requests,Yes,Rechtspraak pages if HTTP 403 prevents live sc...
3,ai_assisted_selected_page_extraction,Constrained LLM-assisted extraction from manua...,Heterogeneous pages where rule-based scraping ...,Yes,Category C party boards; Category H internatio...
4,selenium_rendered_page_extraction,Browser-based collection of a dynamically rend...,Official pages where required content is loade...,Yes,ambassadors; charges_d_affaires


,decision_area,decision,reasoning,implication
0,state_owned_enterprise_board,Excluded for NL,EU/NL list states no qualifying Dutch state en...,No Dutch source collection needed for Category G
1,international_organisation_leadership,Excluded for NL,Operationally complex; excluded as this group ...,No testing done on coverage of Category H


## 3. Scope of the empirical benchmark

The taxonomy workbook contains both a Dutch implementation layer and a preliminary EU-wide extension layer.

Only the Dutch implementation layer is used to construct and evaluate benchmark version 3. The EU-wide coding sheets are retained as a structured design artefact for future implementation and are not used in the empirical matching, coverage, or completeness analyses reported in this study.

The empirical benchmark scope is determined by the inclusion and operationalisation decisions recorded in the `nl_source_mapping_detail` and `scope_decisions` sheets.

In [6]:
# -------------------------------------------------------------------
# Save a reproducibility snapshot of the Dutch implementation inputs
# -------------------------------------------------------------------
# This output records the taxonomy and scope decisions used by the
# final Dutch benchmark pipeline. It does not alter the input workbook.

DUTCH_COLLECTION_AUDIT_PATH = (
    OUTPUT_DIR / "dutch_collection_and_scope_audit.csv"
)

audit_columns = [
    "canonical_category",
    "local_taxonomy_category",
    "role_nl",
    "role_eng",
    "source_url",
    "source_type",
    "extraction_strategy",
    "included_in_main_benchmark",
    "operationalisation_status",
    "notes"
]

dutch_collection_audit_df = nl_source_mapping_df[
    audit_columns
].copy()

dutch_collection_audit_df.to_csv(
    DUTCH_COLLECTION_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved Dutch collection and scope audit:")
print(DUTCH_COLLECTION_AUDIT_PATH)

print("\nIncluded categories:")
display(
    dutch_collection_audit_df[
        ["local_taxonomy_category", "included_in_main_benchmark",
         "operationalisation_status", "extraction_strategy"]
    ]
)

Saved Dutch collection and scope audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\dutch_collection_and_scope_audit.csv

Included categories:


,local_taxonomy_category,included_in_main_benchmark,operationalisation_status,extraction_strategy
0,head_of_state,Yes,Direct,manual_official_source_addition
1,national_executive,Yes,Direct,source_specific_html_selector
2,senate_members,Yes,Direct,source_specific_html_selector
3,house_members,Yes,Direct,source_specific_html_selector
4,registered_party_board,No,Direct,manual_official_source_addition
5,high_judiciary_rvs,Yes,Direct,source_specific_html_selector
6,high_judiciary_hr,Yes,Direct,source_specific_html_selector
7,high_judiciary_cbb,Yes,Direct,manual_official_source_addition
8,high_judiciary_crvb,Yes,Direct,manual_official_source_addition
9,court_of_audit,Yes,Direct,manual_official_source_addition


## 4. Consolidated automated source extraction

This section consolidates the automated extraction steps used in the benchmark-development process.

Three automated collection routes are implemented:

1. generic and source-specific extraction from the official PEP source mapping;
2. a dedicated BeautifulSoup extractor for the official Eerste Kamer member page;
3. Selenium-based extraction from the dynamically rendered Rijksoverheid diplomatic page.

The outputs of this notebook are intermediate candidate-record files and collection logs. Manual official-source additions and manually coded Category C records are retained as separate reviewed inputs and are integrated in the next notebook.

In [7]:
# -------------------------------------------------------------------
# Additional imports for automated source extraction
# -------------------------------------------------------------------

import hashlib
import json
import time

from dataclasses import asdict, dataclass
from typing import List, Optional
from urllib.parse import urlparse, urljoin

from bs4 import BeautifulSoup


# -------------------------------------------------------------------
# Output folders and output-file paths
# -------------------------------------------------------------------
# All generated collection outputs are stored in data/output.
# Raw source material is stored separately in raw_html and raw_text.

COMPONENT_DIR = OUTPUT_DIR / "collection_components"
COMPONENT_DIR.mkdir(parents=True, exist_ok=True)

GENERIC_CANDIDATES_PATH = (
    COMPONENT_DIR / "generic_source_candidates.csv"
)

SENATE_CANDIDATES_PATH = (
    COMPONENT_DIR / "senate_members_candidates.csv"
)

DIPLOMATIC_ALL_CANDIDATES_PATH = (
    COMPONENT_DIR / "diplomatic_candidates_all_pages.csv"
)

DIPLOMATIC_IN_SCOPE_PATH = (
    COMPONENT_DIR / "diplomatic_in_scope_candidates.csv"
)

FETCH_LOG_PATH = OUTPUT_DIR / "collection_fetch_log.csv"
ERROR_LOG_PATH = OUTPUT_DIR / "collection_error_log.csv"
MANUAL_SOURCE_LOG_PATH = OUTPUT_DIR / "manual_source_components_log.csv"
RUN_SUMMARY_PATH = OUTPUT_DIR / "collection_run_summary.json"

# One timestamp is used throughout this notebook run.
COLLECTION_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

print("Collection timestamp (UTC):", COLLECTION_TIMESTAMP_UTC)

print("\nComponent output folder:")
print(COMPONENT_DIR)

print("\nRaw HTML folder:")
print(RAW_HTML_DIR)

print("\nRaw text folder:")
print(RAW_TEXT_DIR)

Collection timestamp (UTC): 2026-06-22T15:02:25.905779+00:00

Component output folder:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components

Raw HTML folder:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\raw_html

Raw text folder:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\raw_text


## 4.1 Common source-record structure

The generic scraper and the source-specific extractors use the same record structure.

Each extracted candidate record retains the source category, role description, source URL, extraction method, and collection timestamp. This makes it possible to trace every automated candidate back to the relevant official source page.

In [8]:
# -------------------------------------------------------------------
# Common data structures for automated extraction
# -------------------------------------------------------------------

@dataclass
class SourceRow:
    """
    One row from PEP_source_mapping.csv.
    """
    source_row: int
    category: str
    role_nl: str
    role_eng: str
    source_text: str
    source_url: Optional[str]


@dataclass
class CandidatePersonRecord:
    """
    One candidate person extracted from an official source page.

    These records are candidates, not automatically confirmed final
    benchmark records. Confirmation and integration occur in notebook 02.
    """
    category: str
    source_row: int
    role_nl: str
    role_eng: str
    person_name: str
    role_title: str
    institution: str
    source_url: str
    page_title: str
    extractor_used: str
    access_timestamp_utc: str

## 4.2 Static official-source collection and candidate extraction

This section runs the static-page extraction routes that were used during benchmark development.

The output consists of candidate records, not automatically confirmed benchmark records. Candidate records are retained with their source metadata and are later reviewed, cleaned, and integrated with manual official-source additions in `02_clean_build_benchmark.ipynb`.

The static collection covers source-mapping Categories A, B, D, and E. Category C is handled through manually reviewed party-board coding, Category F is handled through a dedicated Selenium extraction route, and Categories G and H are outside the empirical benchmark scope.

In [9]:
# -------------------------------------------------------------------
# Static official-source collection helpers
# -------------------------------------------------------------------
# This section is a cleaned version of the original generic scraper.
# It preserves the useful source-specific extractors while keeping
# generic extraction outputs clearly labelled as candidate records.

USER_AGENT = "MScThesisPEPScraper/1.0 (+research use)"
REQUEST_TIMEOUT_SECONDS = 30
SLEEP_SECONDS = 1.0

# Categories collected through this static requests + BeautifulSoup route.
STATIC_SOURCE_CATEGORIES = {"A", "B", "D", "E"}

# These sources are handled later through dedicated extractors.
DEDICATED_SOURCE_URL_MARKERS = [
    "eerstekamer.nl/alle_leden",
    "rijksoverheid.nl/themas/migratie-en-reizen/ambassades-consulaten",
    "rijksoverheid.nl/onderwerpen/ambassades-consulaten",
    "rijksoverheid.nl/themas/migratie-en-reizen/ambassadeurs"
]

STOP_WORDS = {
    "home", "contact", "privacy", "cookies", "sitemap", "vacatures",
    "nieuws", "agenda", "archief", "english", "nederlands", "service",
    "menu", "zoeken", "meer", "lees verder", "download", "organisatie",
    "over ons", "onze mensen", "standpunten", "fracties", "kamerleden"
}

NAVIGATION_PHRASES = {
    "ga direct naar inhoud",
    "direct naar inhoud",
    "naar inhoud",
    "hoofdnavigatie",
    "menu",
    "zoek",
    "search",
    "breadcrumb",
    "toggle navigation"
}

NAME_PARTICLES = {
    "van", "de", "den", "der", "ter", "ten", "te", "het",
    "op", "aan", "von", "zu", "di", "da", "del", "la", "le"
}

ROLE_HINTS = [
    "minister", "staatssecretaris", "president", "vice-president",
    "voorzitter", "lid", "member", "judge", "justice", "director",
    "board", "bestuur", "raad", "commissioner", "governor",
    "commandant", "ambassadeur", "ambassador", "registrar",
    "secretary", "principal", "executive", "management", "leadership",
    "college", "koning", "koningin"
]


def extract_first_url(value):
    """
    Extract a usable HTTP(S) URL from a source-mapping value.

    The original source mapping sometimes contains URLs embedded in
    longer text. This function returns the first usable URL.
    """
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    match = re.search(r"https?://[^\s,;]+", value)

    if match:
        return match.group(0).strip()

    if value.lower().startswith("www."):
        return "https://" + value

    return None


def repair_mojibake(text):
    """
    Repair common UTF-8 / Windows-encoding display corruption.

    This is kept from the original scraper because some official pages
    returned incorrectly decoded accented characters.
    """
    if not text:
        return text

    bad_markers = (
        "Ã", "Å", "â€™", "â€œ",
        "â€", "â€“", "â€”"
    )

    if any(marker in text for marker in bad_markers):
        try:
            return text.encode("latin1").decode("utf-8")
        except Exception:
            return text

    return text


def clean_text(value):
    """
    Convert a value to a trimmed one-line string.
    """
    if pd.isna(value):
        return ""

    return str(value).replace("\n", " ").strip()


def normalise_extracted_name(name):
    """
    Apply lightweight cleaning to a candidate name.

    This does not remove titles yet. Title cleaning is performed in
    notebook 02, where all benchmark components are standardised.
    """
    name = repair_mojibake(clean_text(name))
    name = re.sub(r"\s+", " ", name)

    return name.strip(" ,;:-")


def looks_like_person_name(text):
    """
    Conservative heuristic for likely person names.

    The result remains a candidate record. Final inclusion is decided
    during benchmark construction and validation.
    """
    text = normalise_extracted_name(text)
    text_lower = text.casefold()

    if text == "" or len(text) < 5:
        return False

    if text_lower in STOP_WORDS:
        return False

    if text_lower in NAVIGATION_PHRASES:
        return False

    if any(character.isdigit() for character in text):
        return False

    if any(symbol in text for symbol in ["@", "/", "\\", "|", ">", "<"]):
        return False

    words = text.split()

    if len(words) < 2 or len(words) > 6:
        return False

    capitalised_count = 0

    for word in words:
        clean_word = word.strip(".,;:()[]{}\"'")

        if clean_word == "":
            continue

        if clean_word.casefold() in NAME_PARTICLES:
            continue

        if clean_word[0].isupper():
            capitalised_count += 1

    return capitalised_count >= 2


def page_title_from_soup(soup):
    """
    Return the page title or first heading where available.
    """
    if soup.title and soup.title.get_text(strip=True):
        return soup.title.get_text(" ", strip=True)

    first_heading = soup.find("h1")

    if first_heading:
        return first_heading.get_text(" ", strip=True)

    return ""


def institution_from_url(url):
    """
    Create a simple institution label from the source website domain.
    """
    host = urlparse(url).netloc.lower()
    host = host.removeprefix("www.")

    return host


def get_main_content_soup(soup):
    """
    Retain the main content area of a source page where possible.

    Navigation, headers, footers, scripts, and similar non-content
    page elements are removed before generic candidate extraction.
    """
    main_node = soup.select_one(
        "main, [role='main'], article, #content, .content, .main"
    )

    working_html = str(main_node) if main_node else str(soup)

    working_soup = BeautifulSoup(working_html, "html.parser")

    for unwanted_element in working_soup.select(
        "header, nav, footer, aside, script, style, noscript, form, "
        ".menu, .nav, .breadcrumb, .skip-link, .visually-hidden"
    ):
        unwanted_element.decompose()

    for anchor in working_soup.select("a[href^='#']"):
        anchor.decompose()

    return working_soup


def safe_filename_from_url(source_row, url):
    """
    Create a stable, Windows-safe file name for archived source files.
    """
    parsed_url = urlparse(url)

    slug = (
        parsed_url.netloc
        + parsed_url.path
    ).strip("/")

    slug = slug.replace("/", "_")
    slug = re.sub(r"[^A-Za-z0-9_.-]+", "_", slug)
    slug = slug[:80] or "page"

    url_hash = hashlib.md5(
        url.encode("utf-8")
    ).hexdigest()[:10]

    return f"source_{int(source_row):03d}_{slug}_{url_hash}"


def write_csv_safely(dataframe, output_path):
    """
    Save a CSV file without failing when an earlier version is open
    in Excel.

    If the intended output file is locked, a timestamped fallback file
    is written instead.
    """
    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    except PermissionError:
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%d_%H%M%S")

        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )

        print(
            "Warning: output file appears to be open. "
            f"Writing to {fallback_path.name} instead."
        )

        dataframe.to_csv(
            fallback_path,
            index=False,
            encoding="utf-8-sig"
        )

        return fallback_path


def build_source_rows(source_mapping_dataframe):
    """
    Convert the source mapping into SourceRow objects.
    """
    source_rows = []

    for index, row in source_mapping_dataframe.iterrows():
        source_rows.append(
            SourceRow(
                source_row=index + 2,
                category=clean_text(row.get("Category", "")).upper(),
                role_nl=clean_text(row.get("Role NL", "")),
                role_eng=clean_text(row.get("Role ENG", "")),
                source_text=clean_text(row.get("Source", "")),
                source_url=extract_first_url(row.get("Source", ""))
            )
        )

    return source_rows


def make_requests_session():
    """
    Create a requests session with a stable research User-Agent.
    """
    session = requests.Session()

    session.headers.update({
        "User-Agent": USER_AGENT
    })

    return session


def fetch_static_page(session, source_url):
    """
    Fetch a static official source page.

    Returns the response after checking for HTTP errors.
    """
    response = session.get(
        source_url,
        timeout=REQUEST_TIMEOUT_SECONDS
    )

    response.raise_for_status()

    if (
        not response.encoding
        or response.encoding.lower() == "iso-8859-1"
    ):
        response.encoding = response.apparent_encoding or "utf-8"

    return response


def archive_static_source(source, source_url, html_text):
    """
    Save raw HTML and cleaned main-content text for a static source.
    """
    base_filename = safe_filename_from_url(
        source.source_row,
        source_url
    )

    raw_html_path = RAW_HTML_DIR / f"{base_filename}.html"
    raw_text_path = RAW_TEXT_DIR / f"{base_filename}.txt"

    raw_html_path.write_text(
        html_text,
        encoding="utf-8"
    )

    source_soup = BeautifulSoup(
        html_text,
        "html.parser"
    )

    main_content_soup = get_main_content_soup(source_soup)

    clean_source_text = main_content_soup.get_text(
        "\n",
        strip=True
    )

    raw_text_path.write_text(
        clean_source_text,
        encoding="utf-8"
    )

    return raw_html_path, raw_text_path

In [10]:
# -------------------------------------------------------------------
# Static candidate extractors
# -------------------------------------------------------------------
# These functions retain the tested source-specific logic from the
# original scraper. Generic extraction is used only when no dedicated
# static extractor is available.

def extract_rijksoverheid_bewindspersonen(source, source_url, soup):
    """
    Extract Dutch ministers and state secretaries from the official
    Rijksoverheid bewindspersonen page.
    """
    records = []

    page_title = page_title_from_soup(soup)

    for link in soup.select("main li a, li a"):
        link_text = link.get_text(" ", strip=True)

        parts = re.split(
            r"\s+(?=Minister|Staatssecretaris|Minister-president)",
            link_text,
            maxsplit=1
        )

        if len(parts) != 2:
            continue

        person_name, role_title = (
            parts[0].strip(),
            parts[1].strip()
        )

        if not looks_like_person_name(person_name):
            continue

        records.append(
            CandidatePersonRecord(
                category=source.category,
                source_row=source.source_row,
                role_nl=source.role_nl,
                role_eng=source.role_eng,
                person_name=normalise_extracted_name(person_name),
                role_title=role_title,
                institution="Rijksoverheid",
                source_url=source_url,
                page_title=page_title,
                extractor_used=(
                    "source_specific_rijksoverheid_bewindspersonen"
                ),
                access_timestamp_utc=COLLECTION_TIMESTAMP_UTC
            )
        )

    return records


def extract_tweedekamer_members(source, source_url, soup):
    """
    Extract Tweede Kamer member names from heading elements on the
    official Tweede Kamer member page.
    """
    records = []

    page_title = page_title_from_soup(soup)

    for heading in soup.find_all(["h3", "h4"]):
        person_name = heading.get_text(" ", strip=True)

        if not looks_like_person_name(person_name):
            continue

        records.append(
            CandidatePersonRecord(
                category=source.category,
                source_row=source.source_row,
                role_nl=source.role_nl,
                role_eng=source.role_eng,
                person_name=normalise_extracted_name(person_name),
                role_title=source.role_nl or "Kamerlid",
                institution="Tweede Kamer",
                source_url=source_url,
                page_title=page_title,
                extractor_used=(
                    "source_specific_tweedekamer_member_headings"
                ),
                access_timestamp_utc=COLLECTION_TIMESTAMP_UTC
            )
        )

    return records


def extract_generic_links(source, source_url, soup):
    """
    Extract likely person-name links from the main source content.
    """
    records = []

    main_content_soup = get_main_content_soup(soup)

    page_title = page_title_from_soup(main_content_soup)
    institution = institution_from_url(source_url)

    seen_names = set()

    for link in main_content_soup.find_all("a"):
        person_name = normalise_extracted_name(
            link.get_text(" ", strip=True)
        )

        normalised_key = person_name.casefold()

        if not looks_like_person_name(person_name):
            continue

        if normalised_key in seen_names:
            continue

        parent_text = normalise_extracted_name(
            link.parent.get_text(" ", strip=True)
        ).casefold()

        link_target = (link.get("href") or "").strip()

        if link_target.startswith("#"):
            continue

        # Retain the original pragmatic rule: links near role-related
        # text are more likely to identify a relevant person.
        if not any(
            role_hint in parent_text
            for role_hint in ROLE_HINTS
        ):
            if len(person_name.split()) < 2:
                continue

        seen_names.add(normalised_key)

        records.append(
            CandidatePersonRecord(
                category=source.category,
                source_row=source.source_row,
                role_nl=source.role_nl,
                role_eng=source.role_eng,
                person_name=person_name,
                role_title=source.role_nl,
                institution=institution,
                source_url=source_url,
                page_title=page_title,
                extractor_used="generic_link_candidate_extractor",
                access_timestamp_utc=COLLECTION_TIMESTAMP_UTC
            )
        )

    return records


def extract_generic_lists(source, source_url, soup):
    """
    Fallback extraction for pages where useful names occur in list or
    paragraph text rather than in links.
    """
    records = []

    main_content_soup = get_main_content_soup(soup)

    page_title = page_title_from_soup(main_content_soup)
    institution = institution_from_url(source_url)

    seen_names = set()

    for element in main_content_soup.find_all(["li", "p", "div"]):
        element_text = normalise_extracted_name(
            element.get_text(" ", strip=True)
        )

        element_text_lower = element_text.casefold()

        if element_text == "":
            continue

        if element_text_lower in NAVIGATION_PHRASES:
            continue

        if len(element_text) < 10 or len(element_text) > 300:
            continue

        if not any(
            role_hint in element_text_lower
            for role_hint in ROLE_HINTS
        ):
            continue

        text_parts = re.split(
            r"[,:;()\-–—]",
            element_text
        )

        for text_part in text_parts:
            person_name = normalise_extracted_name(text_part)
            normalised_key = person_name.casefold()

            if not looks_like_person_name(person_name):
                continue

            if normalised_key in seen_names:
                continue

            seen_names.add(normalised_key)

            records.append(
                CandidatePersonRecord(
                    category=source.category,
                    source_row=source.source_row,
                    role_nl=source.role_nl,
                    role_eng=source.role_eng,
                    person_name=person_name,
                    role_title=source.role_nl,
                    institution=institution,
                    source_url=source_url,
                    page_title=page_title,
                    extractor_used="generic_list_candidate_extractor",
                    access_timestamp_utc=COLLECTION_TIMESTAMP_UTC
                )
            )

    return records


def choose_static_extractor(source_url):
    """
    Select a source-specific extractor for known static official pages.

    Returns None when the generic extractor should be used.
    """
    source_url_lower = source_url.lower()

    if "rijksoverheid.nl/regering/bewindspersonen" in source_url_lower:
        return extract_rijksoverheid_bewindspersonen

    if (
        "tweedekamer.nl/kamerleden_en_commissies/alle_kamerleden"
        in source_url_lower
    ):
        return extract_tweedekamer_members

    return None


def is_dedicated_source(source_url):
    """
    Return True when a source is handled later through a dedicated
    Senate or Selenium diplomatic extractor.
    """
    source_url_lower = source_url.lower()

    return any(
        marker in source_url_lower
        for marker in DEDICATED_SOURCE_URL_MARKERS
    )


def deduplicate_candidate_records(records):
    """
    Remove exact duplicates generated within a static extraction run.
    """
    deduplicated_records = {}

    for record in records:
        record_key = (
            record.person_name.casefold(),
            record.role_title.casefold(),
            record.institution.casefold(),
            record.source_url.casefold()
        )

        deduplicated_records[record_key] = record

    return list(deduplicated_records.values())

## 4.3 Run static extraction

The following cell collects static official pages from the source mapping and writes:

- archived raw HTML;
- cleaned source text;
- a static candidate-record file;
- a fetch log;
- an error log.

Dedicated Senate and diplomatic sources are deliberately skipped here because they are handled in the next sections using methods tailored to their page structures.

In [11]:
# -------------------------------------------------------------------
# Run static official-source collection and candidate extraction
# -------------------------------------------------------------------

def run_static_source_collection(source_mapping_dataframe):
    """
    Run the static requests + BeautifulSoup collection route.

    The function processes only Categories A, B, D, and E. It skips
    dedicated Senate and diplomatic sources, which are handled later.
    """
    source_rows = build_source_rows(source_mapping_dataframe)

    session = make_requests_session()

    all_candidate_records = []
    fetch_log_rows = []
    error_log_rows = []

    eligible_source_rows = [
        source
        for source in source_rows
        if source.category in STATIC_SOURCE_CATEGORIES
    ]

    print("Static source rows considered:", len(eligible_source_rows))

    for source in eligible_source_rows:
        source_url = source.source_url

        # Keep invalid URLs visible in the fetch log.
        if source_url is None:
            fetch_log_rows.append({
                "source_row": source.source_row,
                "category": source.category,
                "role_nl": source.role_nl,
                "source_url": pd.NA,
                "collection_status": "skipped_missing_url",
                "status_code": pd.NA,
                "records_extracted": 0,
                "extractor_used": pd.NA,
                "raw_html_path": pd.NA,
                "raw_text_path": pd.NA,
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC
            })

            continue

        # Senate and diplomatic sources use dedicated methods later.
        if is_dedicated_source(source_url):
            fetch_log_rows.append({
                "source_row": source.source_row,
                "category": source.category,
                "role_nl": source.role_nl,
                "source_url": source_url,
                "collection_status": "skipped_dedicated_extractor",
                "status_code": pd.NA,
                "records_extracted": 0,
                "extractor_used": "dedicated_extractor_in_later_section",
                "raw_html_path": pd.NA,
                "raw_text_path": pd.NA,
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC
            })

            continue

        try:
            print(
                f"Collecting source row {source.source_row}: "
                f"{source.role_nl}"
            )

            response = fetch_static_page(
                session,
                source_url
            )

            html_text = repair_mojibake(response.text)

            raw_html_path, raw_text_path = archive_static_source(
                source,
                source_url,
                html_text
            )

            source_soup = BeautifulSoup(
                html_text,
                "html.parser"
            )

            source_specific_extractor = choose_static_extractor(
                source_url
            )

            if source_specific_extractor is not None:
                extracted_records = source_specific_extractor(
                    source,
                    source_url,
                    source_soup
                )
            else:
                extracted_records = extract_generic_links(
                    source,
                    source_url,
                    source_soup
                )

                # Preserve the fallback logic used in the original
                # scraper when link extraction finds little content.
                if len(extracted_records) < 2:
                    extracted_records = extract_generic_lists(
                        source,
                        source_url,
                        source_soup
                    )

            all_candidate_records.extend(extracted_records)

            fetch_log_rows.append({
                "source_row": source.source_row,
                "category": source.category,
                "role_nl": source.role_nl,
                "source_url": source_url,
                "collection_status": "success",
                "status_code": response.status_code,
                "records_extracted": len(extracted_records),
                "extractor_used": (
                    source_specific_extractor.__name__
                    if source_specific_extractor is not None
                    else "generic_static_candidate_extractor"
                ),
                "raw_html_path": str(raw_html_path),
                "raw_text_path": str(raw_text_path),
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC
            })

        except Exception as error:
            error_log_rows.append({
                "source_row": source.source_row,
                "category": source.category,
                "role_nl": source.role_nl,
                "source_url": source_url,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC
            })

            fetch_log_rows.append({
                "source_row": source.source_row,
                "category": source.category,
                "role_nl": source.role_nl,
                "source_url": source_url,
                "collection_status": "error",
                "status_code": pd.NA,
                "records_extracted": 0,
                "extractor_used": pd.NA,
                "raw_html_path": pd.NA,
                "raw_text_path": pd.NA,
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC
            })

        finally:
            time.sleep(SLEEP_SECONDS)

    deduplicated_records = deduplicate_candidate_records(
        all_candidate_records
    )

    candidate_dataframe = pd.DataFrame(
        [asdict(record) for record in deduplicated_records]
    )

    fetch_log_dataframe = pd.DataFrame(fetch_log_rows)
    error_log_dataframe = pd.DataFrame(error_log_rows)

    candidate_output_path = write_csv_safely(
        candidate_dataframe,
        GENERIC_CANDIDATES_PATH
    )

    fetch_log_output_path = write_csv_safely(
        fetch_log_dataframe,
        FETCH_LOG_PATH
    )

    error_log_output_path = write_csv_safely(
        error_log_dataframe,
        ERROR_LOG_PATH
    )

    return {
        "source_rows_considered": len(eligible_source_rows),
        "candidate_records_before_deduplication": len(all_candidate_records),
        "candidate_records_after_deduplication": len(deduplicated_records),
        "candidate_output_path": str(candidate_output_path),
        "fetch_log_output_path": str(fetch_log_output_path),
        "error_log_output_path": str(error_log_output_path),
        "fetch_log_dataframe": fetch_log_dataframe,
        "error_log_dataframe": error_log_dataframe,
        "candidate_dataframe": candidate_dataframe
    }


static_collection_result = run_static_source_collection(
    pep_source_mapping_df
)

print("\nStatic collection summary:")
print(
    "Source rows considered:",
    static_collection_result["source_rows_considered"]
)
print(
    "Candidate records before deduplication:",
    static_collection_result["candidate_records_before_deduplication"]
)
print(
    "Candidate records after deduplication:",
    static_collection_result["candidate_records_after_deduplication"]
)

print("\nCandidate output:")
print(static_collection_result["candidate_output_path"])

print("\nFetch log:")
print(static_collection_result["fetch_log_output_path"])

print("\nError log:")
print(static_collection_result["error_log_output_path"])

Static source rows considered: 12

Static collection summary:
Source rows considered: 12
Candidate records before deduplication: 328
Candidate records after deduplication: 328

Candidate output:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components\generic_source_candidates.csv

Fetch log:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_fetch_log.csv

Error log:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_error_log.csv


In [12]:
static_collection_result["fetch_log_dataframe"]["collection_status"].value_counts()

collection_status
success                        9
error                          2
skipped_dedicated_extractor    1
Name: count, dtype: int64

In [13]:
# -------------------------------------------------------------------
# Review static collection errors and extraction counts
# -------------------------------------------------------------------
# These diagnostics document which official pages could not be collected
# automatically and how many candidate records were produced per source.

print("Static collection errors:")
display(
    static_collection_result["error_log_dataframe"][
        [
            "source_row",
            "category",
            "role_nl",
            "source_url",
            "error_type",
            "error_message"
        ]
    ]
)

print("\nCandidate records by category:")
display(
    static_collection_result["candidate_dataframe"]
    .groupby("category")
    .size()
    .reset_index(name="candidate_records")
)

print("\nRecords extracted per successfully collected source:")
display(
    static_collection_result["fetch_log_dataframe"][
        static_collection_result["fetch_log_dataframe"]["collection_status"]
        .eq("success")
    ][
        [
            "source_row",
            "category",
            "role_nl",
            "records_extracted",
            "extractor_used"
        ]
    ]
)

Static collection errors:


,source_row,category,role_nl,source_url,error_type,error_message
0,26,D,De leden met rechtspraak belast van het Colleg...,https://www.rechtspraak.nl/organisatie-en-cont...,HTTPError,403 Client Error: Forbidden for url: https://w...
1,27,D,De leden met rechtspraak belast van de Central...,https://www.rechtspraak.nl/organisatie-en-cont...,HTTPError,403 Client Error: Forbidden for url: https://w...



Candidate records by category:


,category,candidate_records
0,A,70
1,B,150
2,D,108



Records extracted per successfully collected source:


,source_row,category,role_nl,records_extracted,extractor_used
0,2,A,Koning,0,generic_static_candidate_extractor
1,3,A,Minister president,28,extract_rijksoverheid_bewindspersonen
2,4,A,Ministers,28,extract_rijksoverheid_bewindspersonen
3,5,A,Staatssecretarissen,14,generic_static_candidate_extractor
5,7,B,Leden van de Tweede Kamer der Staten-Generaal,150,extract_tweedekamer_members
6,24,D,"De leden, de staatsraden en de staatsraden in ...",74,generic_static_candidate_extractor
7,25,D,"De president, de vice-presidenten, de raadsher...",34,generic_static_candidate_extractor
10,28,E,De leden in gewone en in buitengewone dienst v...,0,generic_static_candidate_extractor
11,29,E,President en directeuren van de directie van D...,0,generic_static_candidate_extractor


## 4.4 Dedicated Eerste Kamer extraction with profile-name enrichment

The official “Alle leden” page states that the Eerste Kamer has 75 members and provides a personal page for each member. The list-page labels often use initials, titles, and degree suffixes. This extraction therefore:

1. identifies all 75 unique `/persoon/...` profile links from the official member list;
2. retains the original list-page label for auditability;
3. retrieves each official personal page;
4. extracts a full name from the introductory sentence where available;
5. falls back to the list-page label if a profile request or extraction fails.

This design keeps all 75 current members in the candidate component and improves name comparability without dropping members whose profile name is identical to the listing label.


In [14]:
# -------------------------------------------------------------------
# Dedicated Eerste Kamer extraction with profile-name enrichment
# -------------------------------------------------------------------
# The official list page contains 75 current members. Each member card
# links to an official /persoon/... page. The profile introduction
# often provides a full first name where the list page only shows
# initials, titles, or degree suffixes.
#
# The key design choice is that every unique profile link creates one
# benchmark candidate. Profile-name extraction enriches a record but
# never determines whether the record is retained.

EERSTE_KAMER_URL = "https://www.eerstekamer.nl/alle_leden"

SENATE_PROFILE_AUDIT_PATH = (
    COMPONENT_DIR
    / "senate_profile_name_enrichment_audit.csv"
)

SENATE_PROFILE_ERRORS_PATH = (
    OUTPUT_DIR
    / "senate_profile_name_enrichment_errors.csv"
)


def find_senate_source_row(source_mapping_dataframe):
    """
    Find the source-mapping row describing Dutch Senate members.
    """
    senate_rows = source_mapping_dataframe[
        source_mapping_dataframe["Role NL"]
        .astype("string")
        .str.contains(
            "Eerste Kamer",
            case=False,
            na=False
        )
    ].copy()

    if senate_rows.empty:
        raise ValueError(
            "Could not find an Eerste Kamer source row in "
            "PEP_source_mapping.csv."
        )

    senate_row = senate_rows.iloc[0]

    return SourceRow(
        source_row=senate_row.name + 2,
        category=clean_text(senate_row["Category"]).upper(),
        role_nl=clean_text(senate_row["Role NL"]),
        role_eng=clean_text(senate_row["Role ENG"]),
        source_text=clean_text(senate_row["Source"]),
        source_url=EERSTE_KAMER_URL
    )


def extract_listing_label(profile_link):
    """
    Extract the short/formal list-page label from one member link.

    The visible name is normally held in an element with class 'naam'.
    A first-text fallback prevents a member from being dropped if the
    page markup changes slightly.
    """
    name_node = profile_link.select_one(".naam")

    if name_node is not None:
        listing_name = normalise_extracted_name(
            name_node.get_text(
                " ",
                strip=True
            )
        )
    else:
        listing_name = normalise_extracted_name(
            next(
                profile_link.stripped_strings,
                ""
            )
        )

    return listing_name


def extract_senate_listing_rows(list_soup, list_page_url):
    """
    Create one row per unique official personal-profile URL.

    The function deliberately starts from '/persoon/' links rather
    than only 'div.naam' elements. This ensures that all 75 linked
    members are retained even where page layout differs.
    """
    listing_rows = []
    seen_profile_urls = set()

    for profile_link in list_soup.select('a[href*="/persoon/"]'):
        href = (
            profile_link.get("href")
            or ""
        ).strip()

        if href == "":
            continue

        profile_url = urljoin(
            list_page_url,
            href
        )

        if not urlparse(profile_url).path.startswith("/persoon/"):
            continue

        if profile_url in seen_profile_urls:
            continue

        seen_profile_urls.add(profile_url)

        listing_name = extract_listing_label(
            profile_link
        )

        if listing_name == "":
            listing_name = pd.NA

        listing_rows.append({
            "senate_listing_name": listing_name,
            "official_profile_url": profile_url
        })

    listing_df = pd.DataFrame(listing_rows)

    listing_df = (
        listing_df
        .drop_duplicates(
            subset=["official_profile_url"]
        )
        .reset_index(drop=True)
    )

    return listing_df


def extract_profile_full_name(profile_soup):
    """
    Extract a full name from the introductory sentence on an official
    Eerste Kamer profile page.

    The profile text is read as one continuous string because linked
    party names can otherwise split the introduction into separate
    BeautifulSoup text lines.

    Example:
    'Bastiaan van Apeldoorn (1970) is vanaf 9 juni 2015 lid van de
    SP-fractie in de Eerste Kamer.'
    """
    main_content_soup = get_main_content_soup(
        profile_soup
    )

    # Use spaces, not line breaks. This keeps sentences intact when
    # part of the sentence contains a hyperlink, such as "SP-fractie".
    profile_text = normalise_extracted_name(
        main_content_soup.get_text(
            " ",
            strip=True
        )
    )

    # Primary pattern: the standard Eerste Kamer profile introduction
    # contains a full name followed by a birth year and "is".
    profile_intro_pattern = re.compile(
        r"""
        (?P<name>
            [A-ZÀ-ÖØ-Ý]
            [A-Za-zÀ-ÖØ-öø-ÿ'’.\- ]{1,100}?
        )
        \s*
        \(
            \s*\d{4}\s*
        \)
        \s+is\b
        .{0,250}?
        \blid\b
        .{0,250}?
        \bEerste\s+Kamer\b
        """,
        flags=re.IGNORECASE | re.VERBOSE
    )

    match = profile_intro_pattern.search(
        profile_text
    )

    if match is not None:
        extracted_name = normalise_extracted_name(
            match.group("name")
        )

        if looks_like_person_name(extracted_name):
            return extracted_name

    # Fallback pattern for profiles that do not show a birth year in
    # the introductory sentence.
    profile_intro_fallback_pattern = re.compile(
        r"""
        (?P<name>
            [A-ZÀ-ÖØ-Ý]
            [A-Za-zÀ-ÖØ-öø-ÿ'’.\- ]{1,100}?
        )
        \s+is\b
        .{0,250}?
        \blid\b
        .{0,250}?
        \bEerste\s+Kamer\b
        """,
        flags=re.IGNORECASE | re.VERBOSE
    )

    fallback_match = profile_intro_fallback_pattern.search(
        profile_text
    )

    if fallback_match is not None:
        extracted_name = normalise_extracted_name(
            fallback_match.group("name")
        )

        if looks_like_person_name(extracted_name):
            return extracted_name

    return pd.NA


def run_senate_extraction(source_mapping_dataframe):
    """
    Collect all 75 Senate member records and enrich names from official
    personal profile pages where possible.
    """
    senate_source = find_senate_source_row(
        source_mapping_dataframe
    )

    session = make_requests_session()

    # ---------------------------------------------------------------
    # Fetch and archive the official list page.
    # ---------------------------------------------------------------
    list_response = fetch_static_page(
        session,
        senate_source.source_url
    )

    list_html_text = repair_mojibake(
        list_response.text
    )

    list_raw_html_path, list_raw_text_path = (
        archive_static_source(
            senate_source,
            senate_source.source_url,
            list_html_text
        )
    )

    list_soup = BeautifulSoup(
        list_html_text,
        "html.parser"
    )

    listing_df = extract_senate_listing_rows(
        list_soup=list_soup,
        list_page_url=senate_source.source_url
    )

    print(
        "Unique official Senate profile links found:",
        len(listing_df)
    )

    if len(listing_df) != 75:
        raise ValueError(
            "The official Eerste Kamer page should provide 75 current "
            f"member profile links, but {len(listing_df)} were found. "
            "Do not overwrite the Senate component until this is resolved."
        )

    # ---------------------------------------------------------------
    # Fetch profiles. Every list-page record is retained, even if a
    # profile request or full-name extraction fails.
    # ---------------------------------------------------------------
    enriched_rows = []
    error_rows = []

    for _, listing_row in listing_df.iterrows():
        listing_name = listing_row[
            "senate_listing_name"
        ]

        profile_url = listing_row[
            "official_profile_url"
        ]

        profile_name = pd.NA
        profile_title = pd.NA
        profile_fetch_status = "not_requested"
        profile_name_extraction_status = (
            "listing_name_fallback"
        )

        try:
            profile_response = fetch_static_page(
                session,
                profile_url
            )

            profile_html_text = repair_mojibake(
                profile_response.text
            )

            archive_static_source(
                senate_source,
                profile_url,
                profile_html_text
            )

            profile_soup = BeautifulSoup(
                profile_html_text,
                "html.parser"
            )

            profile_title = page_title_from_soup(
                profile_soup
            )

            profile_name = extract_profile_full_name(
                profile_soup
            )

            profile_fetch_status = "success"

            if pd.notna(profile_name):
                profile_name_extraction_status = (
                    "profile_intro_full_name"
                )
            else:
                profile_name_extraction_status = (
                    "profile_name_not_extracted_listing_fallback"
                )

        except Exception as error:
            profile_fetch_status = "error"
            profile_name_extraction_status = (
                "profile_request_or_parse_error_listing_fallback"
            )

            error_rows.append({
                "senate_listing_name": listing_name,
                "official_profile_url": profile_url,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "collection_timestamp_utc": (
                    COLLECTION_TIMESTAMP_UTC
                )
            })

        # Respect the official source during the 75-profile request run.
        time.sleep(SLEEP_SECONDS)

        final_person_name = (
            profile_name
            if pd.notna(profile_name)
            else listing_name
        )

        if pd.isna(final_person_name) or str(final_person_name).strip() == "":
            raise ValueError(
                "A Senate profile link had neither a usable list-page "
                f"name nor a profile-derived name: {profile_url}"
            )

        enriched_rows.append({
            "category": senate_source.category,
            "source_row": senate_source.source_row,
            "role_nl": senate_source.role_nl,
            "role_eng": senate_source.role_eng,
            "person_name": final_person_name,
            "role_title": (
                "Lid van de Eerste Kamer der Staten-Generaal"
            ),
            "institution": (
                "Eerste Kamer der Staten-Generaal"
            ),
            "source_url": profile_url,
            "page_title": profile_title,
            "extractor_used": (
                "source_specific_eerstekamer_listing_profile_enrichment"
            ),
            "access_timestamp_utc": (
                COLLECTION_TIMESTAMP_UTC
            ),
            "senate_listing_name": listing_name,
            "official_profile_name": profile_name,
            "official_profile_url": profile_url,
            "profile_fetch_status": profile_fetch_status,
            "profile_name_extraction_status": (
                profile_name_extraction_status
            ),
            "evidence_text": (
                f"Official list label: {listing_name} | "
                f"Official profile URL: {profile_url} | "
                f"Profile-derived full name: {profile_name}"
            )
        })

    senate_candidates_df = pd.DataFrame(
        enriched_rows
    )

    # Deduplicate only by the official personal page, never by name.
    # This preserves all 75 current offices even when two records have
    # similar formal labels.
    senate_candidates_df = (
        senate_candidates_df
        .drop_duplicates(
            subset=["official_profile_url"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    if len(senate_candidates_df) != 75:
        raise ValueError(
            "Senate extraction must retain 75 member records after "
            f"profile enrichment, but retained {len(senate_candidates_df)}."
        )

    senate_profile_audit_df = senate_candidates_df[
        [
            "senate_listing_name",
            "official_profile_name",
            "person_name",
            "official_profile_url",
            "profile_fetch_status",
            "profile_name_extraction_status"
        ]
    ].copy()

    error_columns = [
        "senate_listing_name",
        "official_profile_url",
        "error_type",
        "error_message",
        "collection_timestamp_utc"
    ]

    senate_profile_errors_df = pd.DataFrame(
        error_rows,
        columns=error_columns
    )

    senate_output_path = write_csv_safely(
        senate_candidates_df,
        SENATE_CANDIDATES_PATH
    )

    profile_audit_output_path = write_csv_safely(
        senate_profile_audit_df,
        SENATE_PROFILE_AUDIT_PATH
    )

    profile_errors_output_path = write_csv_safely(
        senate_profile_errors_df,
        SENATE_PROFILE_ERRORS_PATH
    )

    senate_fetch_log_row = pd.DataFrame([
        {
            "source_row": senate_source.source_row,
            "category": senate_source.category,
            "role_nl": senate_source.role_nl,
            "source_url": senate_source.source_url,
            "collection_status": "success",
            "status_code": list_response.status_code,
            "records_extracted": len(senate_candidates_df),
            "profile_pages_requested": len(listing_df),
            "profile_pages_fetched": int(
                senate_candidates_df[
                    "profile_fetch_status"
                ].eq("success").sum()
            ),
            "profile_full_names_extracted": int(
                senate_candidates_df[
                    "profile_name_extraction_status"
                ].eq("profile_intro_full_name").sum()
            ),
            "profile_errors": len(error_rows),
            "extractor_used": (
                "source_specific_eerstekamer_listing_profile_enrichment"
            ),
            "raw_html_path": str(list_raw_html_path),
            "raw_text_path": str(list_raw_text_path),
            "collection_timestamp_utc": (
                COLLECTION_TIMESTAMP_UTC
            )
        }
    ])

    return {
        "candidate_dataframe": senate_candidates_df,
        "candidate_output_path": str(senate_output_path),
        "profile_audit_output_path": str(
            profile_audit_output_path
        ),
        "profile_errors_output_path": str(
            profile_errors_output_path
        ),
        "fetch_log_row": senate_fetch_log_row
    }


senate_extraction_result = run_senate_extraction(
    pep_source_mapping_df
)

senate_candidates_df = senate_extraction_result[
    "candidate_dataframe"
]

print("\nSenate member candidates retained:", len(senate_candidates_df))

print("\nProfile-name extraction status:")
display(
    senate_candidates_df[
        "profile_name_extraction_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "profile_name_extraction_status"
    )
    .reset_index(name="record_count")
)

print("\nVerification: Van Apeldoorn")
display(
    senate_candidates_df[
        senate_candidates_df[
            "official_profile_url"
        ]
        .astype("string")
        .str.contains(
            "van_apeldoorn",
            case=False,
            na=False
        )
    ][
        [
            "senate_listing_name",
            "official_profile_name",
            "person_name",
            "official_profile_url",
            "profile_name_extraction_status"
        ]
    ]
)

print("\nSaved Senate candidate file:")
print(senate_extraction_result["candidate_output_path"])

print("\nSaved Senate profile-name audit:")
print(senate_extraction_result["profile_audit_output_path"])


Unique official Senate profile links found: 75

Senate member candidates retained: 75

Profile-name extraction status:


,profile_name_extraction_status,record_count
0,profile_intro_full_name,73
1,profile_name_not_extracted_listing_fallback,2



Verification: Van Apeldoorn


,senate_listing_name,official_profile_name,person_name,official_profile_url,profile_name_extraction_status
1,prof. dr. E.B. van Apeldoorn,Bastiaan van Apeldoorn,Bastiaan van Apeldoorn,https://www.eerstekamer.nl/persoon/prof_dr_e_b...,profile_intro_full_name



Saved Senate candidate file:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components\senate_members_candidates.csv

Saved Senate profile-name audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components\senate_profile_name_enrichment_audit.csv


In [15]:
# -------------------------------------------------------------------
# Diagnose Senate profile-name enrichment
# -------------------------------------------------------------------

print("Total Senate records:", len(senate_candidates_df))

print("\nProfile-name extraction status:")
display(
    senate_candidates_df[
        "profile_name_extraction_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "profile_name_extraction_status"
    )
    .reset_index(
        name="record_count"
    )
)

print("\nRows where the extractor fell back to the list-page name:")
display(
    senate_candidates_df[
        senate_candidates_df[
            "profile_name_extraction_status"
        ]
        .ne("profile_intro_full_name")
    ][
        [
            "senate_listing_name",
            "official_profile_name",
            "person_name",
            "official_profile_url",
            "profile_fetch_status",
            "profile_name_extraction_status"
        ]
    ]
    .sort_values("senate_listing_name")
)

print("\nSpecific check: Bastiaan van Apeldoorn")
display(
    senate_candidates_df[
        senate_candidates_df[
            "official_profile_url"
        ]
        .astype("string")
        .str.contains(
            "van_apeldoorn",
            case=False,
            na=False
        )
    ][
        [
            "senate_listing_name",
            "official_profile_name",
            "person_name",
            "official_profile_url",
            "profile_fetch_status",
            "profile_name_extraction_status"
        ]
    ]
)

Total Senate records: 75

Profile-name extraction status:


,profile_name_extraction_status,record_count
0,profile_intro_full_name,73
1,profile_name_not_extracted_listing_fallback,2



Rows where the extractor fell back to the list-page name:


,senate_listing_name,official_profile_name,person_name,official_profile_url,profile_fetch_status,profile_name_extraction_status
32,A.J.M. van Kesteren,NaN,A.J.M. van Kesteren,https://www.eerstekamer.nl/persoon/a_j_m_van_k...,success,profile_name_not_extracted_listing_fallback
9,mr. R.S. Croll,NaN,mr. R.S. Croll,https://www.eerstekamer.nl/persoon/mr_r_s_crol...,success,profile_name_not_extracted_listing_fallback



Specific check: Bastiaan van Apeldoorn


,senate_listing_name,official_profile_name,person_name,official_profile_url,profile_fetch_status,profile_name_extraction_status
1,prof. dr. E.B. van Apeldoorn,Bastiaan van Apeldoorn,Bastiaan van Apeldoorn,https://www.eerstekamer.nl/persoon/prof_dr_e_b...,success,profile_intro_full_name


## 4.5 Selenium extraction of diplomatic roles

The official Rijksoverheid diplomatic page loads its results dynamically. The relevant names are not reliably available in the initial HTML returned by `requests`.

This section therefore uses Selenium to render the official page in a browser, navigate through all result pages, and extract visible diplomatic records.

Only two categories are included in the final benchmark:

- ambassadors;
- chargés d'affaires.

Other diplomatic roles, such as consuls-general, permanent representatives, and special envoys, are retained in the all-candidates output but excluded from the final benchmark in notebook 02.

In [16]:
# -------------------------------------------------------------------
# Selenium imports and diplomatic-source configuration
# -------------------------------------------------------------------

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException
)
from webdriver_manager.chrome import ChromeDriverManager


# -------------------------------------------------------------------
# Find the official diplomatic source URL and source-mapping metadata
# -------------------------------------------------------------------

diplomatic_taxonomy_df = nl_source_mapping_df[
    nl_source_mapping_df["local_taxonomy_category"].isin(
        [
            "ambassadors",
            "charges_d_affaires"
        ]
    )
].copy()

if diplomatic_taxonomy_df.empty:
    raise ValueError(
        "Could not find ambassadors and charges_d_affaires in "
        "nl_source_mapping_detail."
    )

DIPLOMATIC_URL = (
    diplomatic_taxonomy_df["source_url"]
    .dropna()
    .iloc[0]
)

print("Diplomatic source URL:")
print(DIPLOMATIC_URL)


def find_source_mapping_row_for_text(
    source_mapping_dataframe,
    search_text
):
    """
    Find one source-mapping row based on text in the Dutch role label.

    This links each extracted diplomatic record back to the correct
    ambassador or chargé d'affaires source-mapping entry.
    """
    matching_rows = source_mapping_dataframe[
        source_mapping_dataframe["Role NL"]
        .astype("string")
        .str.contains(
            search_text,
            case=False,
            na=False
        )
    ].copy()

    if matching_rows.empty:
        raise ValueError(
            f"Could not find a source-mapping row containing: {search_text}"
        )

    row = matching_rows.iloc[0]

    return SourceRow(
        source_row=row.name + 2,
        category=clean_text(row["Category"]).upper(),
        role_nl=clean_text(row["Role NL"]),
        role_eng=clean_text(row["Role ENG"]),
        source_text=clean_text(row["Source"]),
        source_url=DIPLOMATIC_URL
    )


AMBASSADOR_SOURCE = find_source_mapping_row_for_text(
    pep_source_mapping_df,
    "Ambassadeur"
)

CHARGE_SOURCE = find_source_mapping_row_for_text(
    pep_source_mapping_df,
    "Zaakgelastigde"
)

print("\nAmbassador source row:", AMBASSADOR_SOURCE.source_row)
print("Chargé d'affaires source row:", CHARGE_SOURCE.source_row)

Diplomatic source URL:
https://www.rijksoverheid.nl/onderwerpen/ambassades-consulaten-en-overige-vertegenwoordigingen/ambassadeurs

Ambassador source row: 30
Chargé d'affaires source row: 31


### Rendered-page extraction helpers

The helper functions below:

1. open the Rijksoverheid page in Chrome;
2. wait for visible rendered content;
3. extract repeated `name → role → Lees verder` records;
4. move through each result page;
5. archive both rendered text and page HTML for every page;
6. save all diplomatic candidates and the benchmark-relevant subset.

In [17]:
# -------------------------------------------------------------------
# Selenium diplomatic extraction helpers
# -------------------------------------------------------------------

DIPLOMATIC_WAIT_SECONDS = 20
MAX_DIPLOMATIC_RESULT_PAGES = 30


def start_chrome_driver():
    """
    Start a Chrome browser controlled by Selenium.

    webdriver-manager downloads or selects a ChromeDriver version that
    matches the installed Google Chrome browser.
    """
    chrome_options = Options()

    chrome_options.add_argument("--window-size=1400,1000")
    chrome_options.add_argument("--disable-notifications")

    return webdriver.Chrome(
        service=Service(
            ChromeDriverManager().install()
        ),
        options=chrome_options
    )


def get_rendered_page_text(driver):
    """
    Return the currently visible text from the rendered browser page.
    """
    return driver.find_element(
        By.TAG_NAME,
        "body"
    ).text


def archive_rendered_diplomatic_page(
    page_number,
    page_html,
    rendered_text
):
    """
    Archive HTML and visible text for one rendered diplomatic result page.
    """
    collection_date = COLLECTION_TIMESTAMP_UTC[:10].replace("-", "")

    base_filename = (
        f"diplomatic_page_{page_number:02d}_{collection_date}"
    )

    raw_html_path = RAW_HTML_DIR / f"{base_filename}.html"
    raw_text_path = RAW_TEXT_DIR / f"{base_filename}.txt"

    raw_html_path.write_text(
        page_html,
        encoding="utf-8"
    )

    raw_text_path.write_text(
        rendered_text,
        encoding="utf-8"
    )

    return raw_html_path, raw_text_path


def classify_diplomatic_role(role_title):
    """
    Classify one visible diplomatic role into benchmark categories.

    Returns:
    - taxonomy_category
    - include_in_main_benchmark
    - matching source metadata
    """
    role_lower = role_title.casefold()

    if (
        "zaakgelastigde" in role_lower
        or "chargé" in role_lower
        or "charge d'affaires" in role_lower
    ):
        return (
            "charges_d_affaires",
            "Yes",
            CHARGE_SOURCE
        )

    if "ambassadeur" in role_lower:
        return (
            "ambassadors",
            "Yes",
            AMBASSADOR_SOURCE
        )

    return (
        "other_diplomatic_role",
        "No",
        None
    )


def extract_diplomatic_records_from_text(
    rendered_text,
    page_number
):
    """
    Extract visible diplomatic result cards from rendered page text.

    Expected repeated pattern:

    person name
    diplomatic role and post
    Lees verder
    """
    lines = [
        line.strip()
        for line in rendered_text.splitlines()
        if line.strip()
    ]

    extracted_rows = []

    for line_index in range(len(lines) - 2):
        possible_name = lines[line_index]
        possible_role = lines[line_index + 1]
        possible_marker = lines[line_index + 2]

        if possible_marker.casefold() != "lees verder":
            continue

        role_lower = possible_role.casefold()

        diplomatic_role_terms = [
            "ambassadeur",
            "zaakgelastigde",
            "chargé",
            "charge",
            "consul-generaal",
            "permanent vertegenwoordiger",
            "speciaal gezant"
        ]

        if not any(
            term in role_lower
            for term in diplomatic_role_terms
        ):
            continue

        (
            taxonomy_category,
            include_in_main_benchmark,
            source_metadata
        ) = classify_diplomatic_role(
            possible_role
        )

        if source_metadata is not None:
            source_row = source_metadata.source_row
            role_nl = source_metadata.role_nl
            role_eng = source_metadata.role_eng
            category = source_metadata.category
        else:
            source_row = pd.NA
            role_nl = "Other diplomatic role"
            role_eng = "Other diplomatic role"
            category = "F"

        extracted_rows.append({
            "page_number": page_number,
            "category": category,
            "source_row": source_row,
            "role_nl": role_nl,
            "role_eng": role_eng,
            "person_name": normalise_extracted_name(possible_name),
            "role_title": normalise_extracted_name(possible_role),
            "taxonomy_category": taxonomy_category,
            "include_in_main_benchmark": include_in_main_benchmark,
            "institution": "Ministerie van Buitenlandse Zaken",
            "source_url": DIPLOMATIC_URL,
            "page_title": "Ambassadeurs | Rijksoverheid.nl",
            "extractor_used": "selenium_rendered_page_extraction",
            "access_timestamp_utc": COLLECTION_TIMESTAMP_UTC,
            "evidence_text": (
                f"{possible_name} | {possible_role}"
            )
        })

    return pd.DataFrame(extracted_rows)


def find_visible_next_page_button(driver):
    """
    Find the visible 'Volgende pagina' navigation control.

    Returns None when no next-page control is available.
    """
    candidate_buttons = driver.find_elements(
        By.XPATH,
        (
            "//*[self::a or self::button]"
            "[contains(normalize-space(.), 'Volgende pagina')]"
        )
    )

    for button in candidate_buttons:
        if button.is_displayed() and button.is_enabled():
            return button

    return None


def run_diplomatic_selenium_extraction():
    """
    Render and extract all diplomatic result pages.

    The function archives every rendered page and saves:
    - all diplomatic candidates;
    - only ambassadors and chargés d'affaires.
    """
    driver = None

    all_page_dataframes = []
    diplomatic_fetch_log_rows = []

    try:
        driver = start_chrome_driver()

        driver.get(DIPLOMATIC_URL)

        WebDriverWait(
            driver,
            DIPLOMATIC_WAIT_SECONDS
        ).until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "body")
            )
        )

        time.sleep(3)

        for page_number in range(
            1,
            MAX_DIPLOMATIC_RESULT_PAGES + 1
        ):
            rendered_text = get_rendered_page_text(driver)
            page_html = driver.page_source

            raw_html_path, raw_text_path = (
                archive_rendered_diplomatic_page(
                    page_number=page_number,
                    page_html=page_html,
                    rendered_text=rendered_text
                )
            )

            page_candidates_df = (
                extract_diplomatic_records_from_text(
                    rendered_text=rendered_text,
                    page_number=page_number
                )
            )

            all_page_dataframes.append(page_candidates_df)

            diplomatic_fetch_log_rows.append({
                "source_row": pd.NA,
                "category": "F",
                "role_nl": (
                    "Ambassadeurs en zaakgelastigden"
                ),
                "source_url": DIPLOMATIC_URL,
                "collection_status": "success",
                "status_code": pd.NA,
                "records_extracted": len(page_candidates_df),
                "extractor_used": (
                    "selenium_rendered_page_extraction"
                ),
                "raw_html_path": str(raw_html_path),
                "raw_text_path": str(raw_text_path),
                "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC,
                "page_number": page_number
            })

            print(
                f"Page {page_number}: "
                f"{len(page_candidates_df)} diplomatic candidates"
            )

            next_page_button = find_visible_next_page_button(
                driver
            )

            if next_page_button is None:
                print(
                    "No visible next-page control found. "
                    "Diplomatic extraction complete."
                )
                break

            previous_rendered_text = rendered_text

            driver.execute_script(
                "arguments[0].click();",
                next_page_button
            )

            try:
                WebDriverWait(
                    driver,
                    DIPLOMATIC_WAIT_SECONDS
                ).until(
                    lambda active_driver: (
                        get_rendered_page_text(active_driver)
                        != previous_rendered_text
                    )
                )

            except TimeoutException:
                print(
                    "Page content did not change after clicking "
                    "the next-page control. Stopping extraction."
                )
                break

            time.sleep(1)

        if len(all_page_dataframes) == 0:
            diplomatic_all_candidates_df = pd.DataFrame()
        else:
            diplomatic_all_candidates_df = pd.concat(
                all_page_dataframes,
                ignore_index=True
            )

        diplomatic_all_candidates_df = (
            diplomatic_all_candidates_df
            .drop_duplicates(
                subset=[
                    "person_name",
                    "role_title",
                    "taxonomy_category"
                ]
            )
            .reset_index(drop=True)
        )

        diplomatic_in_scope_df = (
            diplomatic_all_candidates_df[
                diplomatic_all_candidates_df[
                    "include_in_main_benchmark"
                ].eq("Yes")
            ]
            .copy()
            .reset_index(drop=True)
        )

        all_candidates_output_path = write_csv_safely(
            diplomatic_all_candidates_df,
            DIPLOMATIC_ALL_CANDIDATES_PATH
        )

        in_scope_output_path = write_csv_safely(
            diplomatic_in_scope_df,
            DIPLOMATIC_IN_SCOPE_PATH
        )

        diplomatic_fetch_log_df = pd.DataFrame(
            diplomatic_fetch_log_rows
        )

        return {
            "all_candidates_dataframe": diplomatic_all_candidates_df,
            "in_scope_dataframe": diplomatic_in_scope_df,
            "fetch_log_dataframe": diplomatic_fetch_log_df,
            "all_candidates_output_path": str(
                all_candidates_output_path
            ),
            "in_scope_output_path": str(
                in_scope_output_path
            )
        }

    finally:
        if driver is not None:
            driver.quit()
            print("Selenium browser closed.")

### Run the diplomatic extraction

Running the next cell opens a Chrome browser briefly. Do not close the browser manually while the code is running.

The output should contain roughly the same categories as the earlier benchmark-development run. Exact counts may differ if the official source has changed since the original collection date.

In [18]:
# -------------------------------------------------------------------
# Run Selenium diplomatic extraction
# -------------------------------------------------------------------

diplomatic_extraction_result = (
    run_diplomatic_selenium_extraction()
)

diplomatic_all_candidates_df = (
    diplomatic_extraction_result["all_candidates_dataframe"]
)

diplomatic_in_scope_df = (
    diplomatic_extraction_result["in_scope_dataframe"]
)

print("\nTotal diplomatic candidate records:")
print(len(diplomatic_all_candidates_df))

print("\nDiplomatic candidate records by category:")
display(
    diplomatic_all_candidates_df[
        "taxonomy_category"
    ]
    .value_counts()
    .rename_axis("taxonomy_category")
    .reset_index(name="record_count")
)

print("\nIn-scope diplomatic records:")
print(len(diplomatic_in_scope_df))

print("\nIn-scope diplomatic records by category:")
display(
    diplomatic_in_scope_df[
        "taxonomy_category"
    ]
    .value_counts()
    .rename_axis("taxonomy_category")
    .reset_index(name="record_count")
)

print("\nSaved all diplomatic candidates:")
print(
    diplomatic_extraction_result[
        "all_candidates_output_path"
    ]
)

print("\nSaved in-scope diplomatic candidates:")
print(
    diplomatic_extraction_result[
        "in_scope_output_path"
    ]
)

Page 1: 12 diplomatic candidates
Page 2: 12 diplomatic candidates
Page 3: 12 diplomatic candidates
Page 4: 12 diplomatic candidates
Page 5: 11 diplomatic candidates
Page 6: 12 diplomatic candidates
Page 7: 11 diplomatic candidates
Page 8: 12 diplomatic candidates
Page 9: 11 diplomatic candidates
Page 10: 10 diplomatic candidates
Page 11: 0 diplomatic candidates
No visible next-page control found. Diplomatic extraction complete.
Selenium browser closed.

Total diplomatic candidate records:
115

Diplomatic candidate records by category:


,taxonomy_category,record_count
0,ambassadors,82
1,other_diplomatic_role,26
2,charges_d_affaires,7



In-scope diplomatic records:
89

In-scope diplomatic records by category:


,taxonomy_category,record_count
0,ambassadors,82
1,charges_d_affaires,7



Saved all diplomatic candidates:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components\diplomatic_candidates_all_pages.csv

Saved in-scope diplomatic candidates:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_components\diplomatic_in_scope_candidates.csv


## 5. Reviewed manual official-source components

Some benchmark components were not produced through automated extraction because the sources were small, access-restricted, or insufficiently reliable for generic scraping.

These components are retained as versioned reviewed inputs:

- the curated benchmark v1 baseline;
- small official-source additions for the head of state, Court of Audit, and central-bank board;
- manually reviewed CBB, CRvB, and senior-military additions;
- manually coded and reviewed Category C national party-board records.

The files are not hidden exceptions. They are explicit, auditable benchmark components that are combined with automated outputs in `02_clean_build_benchmark.ipynb`.

In [19]:
# -------------------------------------------------------------------
# Create or load the small reviewed official-source additions file
# -------------------------------------------------------------------
# This file contains nine small, manually verified additions that were
# used in benchmark v2:
#
# - one head-of-state record;
# - three Netherlands Court of Audit records;
# - five De Nederlandsche Bank board records.
#
# The file is saved in data/input so it remains a versioned research
# input when the project is uploaded to GitHub.

MANUAL_SMALL_ADDITIONS_PATH = (
    INPUT_DIR / "manual_small_official_additions.csv"
)

if not MANUAL_SMALL_ADDITIONS_PATH.exists():

    manual_small_rows = [
        {
            "person_name": "Koning Willem-Alexander",
            "role_title": "Koning der Nederlanden",
            "role_nl": "Koning",
            "role_eng": "King",
            "taxonomy_category": "head_of_state",
            "official_function": "Head of state",
            "institutional_level": "National monarchy",
            "institution": "Het Koninklijk Huis",
            "source_url": (
                "https://www.koninklijkhuis.nl/"
                "leden-koninklijk-huis/koning-willem-alexander"
            ),
            "page_title": (
                "Koning Willem-Alexander | Het Koninklijk Huis"
            ),
            "extractor_used": "manual_official_source_addition",
            "validation_status": "confirmed",
            "included_in_main_benchmark": "Yes",
            "operationalisation_status": "Direct",
            "source_row": pd.NA,
            "category": "A",
            "cleaning_notes": (
                "Small official-source category manually reviewed "
                "during benchmark construction."
            )
        },

        {
            "person_name": "Pieter Duisenberg",
            "role_title": "President van de Algemene Rekenkamer",
            "role_nl": (
                "Leden in gewone en buitengewone dienst van "
                "de Algemene Rekenkamer"
            ),
            "role_eng": "Members of the Netherlands Court of Audit",
            "taxonomy_category": "court_of_audit",
            "official_function": (
                "Members of the Netherlands Court of Audit"
            ),
            "institutional_level": "National audit institution",
            "institution": "Algemene Rekenkamer",
            "source_url": (
                "https://www.rekenkamer.nl/over-de-algemene-"
                "rekenkamer/organisatie/college"
            ),
            "page_title": "College | Algemene Rekenkamer",
            "extractor_used": "manual_official_source_addition",
            "validation_status": "confirmed",
            "included_in_main_benchmark": "Yes",
            "operationalisation_status": "Direct",
            "source_row": pd.NA,
            "category": "E",
            "cleaning_notes": (
                "Small official-source category manually reviewed "
                "during benchmark construction."
            )
        },

        {
            "person_name": "Ewout Irrgang",
            "role_title": "Collegelid van de Algemene Rekenkamer",
            "role_nl": (
                "Leden in gewone en buitengewone dienst van "
                "de Algemene Rekenkamer"
            ),
            "role_eng": "Members of the Netherlands Court of Audit",
            "taxonomy_category": "court_of_audit",
            "official_function": (
                "Members of the Netherlands Court of Audit"
            ),
            "institutional_level": "National audit institution",
            "institution": "Algemene Rekenkamer",
            "source_url": (
                "https://www.rekenkamer.nl/over-de-algemene-"
                "rekenkamer/organisatie/college"
            ),
            "page_title": "College | Algemene Rekenkamer",
            "extractor_used": "manual_official_source_addition",
            "validation_status": "confirmed",
            "included_in_main_benchmark": "Yes",
            "operationalisation_status": "Direct",
            "source_row": pd.NA,
            "category": "E",
            "cleaning_notes": (
                "Small official-source category manually reviewed "
                "during benchmark construction."
            )
        },

        {
            "person_name": "Barbara Joziasse",
            "role_title": "Collegelid van de Algemene Rekenkamer",
            "role_nl": (
                "Leden in gewone en buitengewone dienst van "
                "de Algemene Rekenkamer"
            ),
            "role_eng": "Members of the Netherlands Court of Audit",
            "taxonomy_category": "court_of_audit",
            "official_function": (
                "Members of the Netherlands Court of Audit"
            ),
            "institutional_level": "National audit institution",
            "institution": "Algemene Rekenkamer",
            "source_url": (
                "https://www.rekenkamer.nl/over-de-algemene-"
                "rekenkamer/organisatie/college"
            ),
            "page_title": "College | Algemene Rekenkamer",
            "extractor_used": "manual_official_source_addition",
            "validation_status": "confirmed",
            "included_in_main_benchmark": "Yes",
            "operationalisation_status": "Direct",
            "source_row": pd.NA,
            "category": "E",
            "cleaning_notes": (
                "Small official-source category manually reviewed "
                "during benchmark construction."
            )
        }
    ]

    dnb_directie_names = [
        (
            "Olaf Sleijpen",
            "President van De Nederlandsche Bank"
        ),
        (
            "Steven Maijoor",
            "Toezichtdirecteur en voorzitter toezicht"
        ),
        (
            "Gita Salden",
            "Toezichtdirecteur"
        ),
        (
            "Cindy van Oorschot",
            "Directeur Intern Bedrijf en Resolutie"
        ),
        (
            "Bas ter Weel",
            "Directeur Monetaire Zaken"
        )
    ]

    for person_name, role_title in dnb_directie_names:
        manual_small_rows.append({
            "person_name": person_name,
            "role_title": role_title,
            "role_nl": (
                "President en directeuren van de directie van "
                "De Nederlandsche Bank"
            ),
            "role_eng": (
                "President and directors of the Executive Board "
                "of De Nederlandsche Bank"
            ),
            "taxonomy_category": "central_bank_board",
            "official_function": (
                "President and directors of De Nederlandsche Bank"
            ),
            "institutional_level": "National central bank",
            "institution": "De Nederlandsche Bank",
            "source_url": "https://www.dnb.nl/over-ons/directie/",
            "page_title": "Directie | De Nederlandsche Bank",
            "extractor_used": "manual_official_source_addition",
            "validation_status": "confirmed",
            "included_in_main_benchmark": "Yes",
            "operationalisation_status": "Direct",
            "source_row": pd.NA,
            "category": "E",
            "cleaning_notes": (
                "Small official-source category manually reviewed "
                "during benchmark construction."
            )
        })

    manual_small_created_df = pd.DataFrame(manual_small_rows)

    manual_small_created_df.to_csv(
        MANUAL_SMALL_ADDITIONS_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("Created manual small-additions input file:")
    print(MANUAL_SMALL_ADDITIONS_PATH)

manual_small_additions_df, manual_small_encoding = (
    read_csv_with_fallback_encodings(
        MANUAL_SMALL_ADDITIONS_PATH
    )
)

print("\nManual small-additions records:", len(manual_small_additions_df))

display(
    manual_small_additions_df[
        [
            "person_name",
            "taxonomy_category",
            "institution",
            "validation_status"
        ]
    ]
)

Loaded manual_small_official_additions.csv using encoding: utf-8

Manual small-additions records: 9


,person_name,taxonomy_category,institution,validation_status
0,Koning Willem-Alexander,head_of_state,Het Koninklijk Huis,confirmed
1,Pieter Duisenberg,court_of_audit,Algemene Rekenkamer,confirmed
2,Ewout Irrgang,court_of_audit,Algemene Rekenkamer,confirmed
3,Barbara Joziasse,court_of_audit,Algemene Rekenkamer,confirmed
4,Olaf Sleijpen,central_bank_board,De Nederlandsche Bank,confirmed
5,Steven Maijoor,central_bank_board,De Nederlandsche Bank,confirmed
6,Gita Salden,central_bank_board,De Nederlandsche Bank,confirmed
7,Cindy van Oorschot,central_bank_board,De Nederlandsche Bank,confirmed
8,Bas ter Weel,central_bank_board,De Nederlandsche Bank,confirmed


## 6. Save collection logs and reproducibility manifest

This section combines the automated collection logs and documents all reviewed manual components.

The resulting manifest provides a concise overview of the inputs and extraction outputs used to build benchmark version 3 in the next notebook.

In [20]:
# -------------------------------------------------------------------
# Load reviewed workbook additions and document manual components
# -------------------------------------------------------------------

manual_workbook_component_rows = []

for sheet_name in manual_additions_sheet_names:

    sheet_df = pd.read_excel(
        MANUAL_ADDITIONS_PATH,
        sheet_name=sheet_name
    )

    taxonomy_categories = (
        sheet_df["taxonomy_category"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
        if "taxonomy_category" in sheet_df.columns
        else []
    )

    manual_workbook_component_rows.append({
        "component_name": f"manual_workbook_{sheet_name}",
        "component_type": "reviewed_manual_official_source_input",
        "records": len(sheet_df),
        "taxonomy_categories": "; ".join(taxonomy_categories),
        "input_or_output_path": str(MANUAL_ADDITIONS_PATH),
        "collection_or_review_method": (
            "manual_official_source_addition"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "Reviewed official-source additions stored in a "
            "versioned Excel worksheet."
        )
    })

manual_v2_additions_total = sum(
    row["records"]
    for row in manual_workbook_component_rows
)

collection_component_rows = [
    {
        "component_name": "curated_benchmark_v1_baseline",
        "component_type": "reviewed_intermediate_input",
        "records": len(curated_v1_df),
        "taxonomy_categories": (
            "national_executive; house_members; "
            "high_judiciary_rvs; high_judiciary_hr"
        ),
        "input_or_output_path": str(CURATED_V1_PATH),
        "collection_or_review_method": (
            "initial_collection_followed_by_manual_screening"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "Frozen reviewed baseline retained because live source "
            "pages and generic extraction behaviour may change."
        )
    },
    {
        "component_name": "generic_static_candidates",
        "component_type": "automated_candidate_output",
        "records": len(static_collection_result["candidate_dataframe"]),
        "taxonomy_categories": "A; B; D; E",
        "input_or_output_path": (
            static_collection_result["candidate_output_path"]
        ),
        "collection_or_review_method": (
            "requests_beautifulsoup_generic_and_source_specific"
        ),
        "used_in_final_benchmark": "No",
        "notes": (
            "Retained as an auditable candidate output. The final "
            "baseline uses the curated reviewed v1 input."
        )
    },
    {
        "component_name": "senate_members",
        "component_type": "automated_candidate_output",
        "records": len(
            senate_extraction_result["candidate_dataframe"]
        ),
        "taxonomy_categories": "senate_members",
        "input_or_output_path": (
            senate_extraction_result["candidate_output_path"]
        ),
        "collection_or_review_method": (
            "source_specific_eerstekamer_listing_profile_enrichment"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "Official Eerste Kamer list extraction with profile-derived full-name enrichment and list-name fallback."
        )
    },
    {
        "component_name": "diplomatic_in_scope_records",
        "component_type": "automated_candidate_output",
        "records": len(diplomatic_in_scope_df),
        "taxonomy_categories": (
            "ambassadors; charges_d_affaires"
        ),
        "input_or_output_path": (
            diplomatic_extraction_result[
                "in_scope_output_path"
            ]
        ),
        "collection_or_review_method": (
            "selenium_rendered_page_extraction"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "Only ambassadors and chargés d'affaires are included. "
            "Other diplomatic roles are retained separately."
        )
    },
    {
        "component_name": "manual_small_official_additions",
        "component_type": "reviewed_manual_official_source_input",
        "records": len(manual_small_additions_df),
        "taxonomy_categories": (
            "head_of_state; court_of_audit; "
            "central_bank_board"
        ),
        "input_or_output_path": str(
            MANUAL_SMALL_ADDITIONS_PATH
        ),
        "collection_or_review_method": (
            "manual_official_source_addition"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "Small categories manually verified from official "
            "institutional sources."
        )
    },
    {
        "component_name": "manual_category_c_party_boards",
        "component_type": "reviewed_manual_official_source_input",
        "records": len(category_c_manual_df),
        "taxonomy_categories": "registered_party_board",
        "input_or_output_path": str(CATEGORY_C_MANUAL_PATH),
        "collection_or_review_method": (
            "manual_official_source_coding"
        ),
        "used_in_final_benchmark": "Yes",
        "notes": (
            "National party-board members manually coded and "
            "reviewed from selected official party sources."
        )
    }
]

collection_component_rows.extend(
    manual_workbook_component_rows
)

collection_component_audit_df = pd.DataFrame(
    collection_component_rows
)

MANUAL_SOURCE_LOG_PATH = (
    OUTPUT_DIR / "manual_source_components_log.csv"
)

manual_source_log_output_path = write_csv_safely(
    collection_component_audit_df,
    MANUAL_SOURCE_LOG_PATH
)

print("Saved component audit:")
print(manual_source_log_output_path)

display(
    collection_component_audit_df[
        [
            "component_name",
            "records",
            "taxonomy_categories",
            "used_in_final_benchmark"
        ]
    ]
)

Saved component audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\manual_source_components_log.csv


,component_name,records,taxonomy_categories,used_in_final_benchmark
0,curated_benchmark_v1_baseline,286,national_executive; house_members; high_judici...,Yes
1,generic_static_candidates,328,A; B; D; E,No
2,senate_members,75,senate_members,Yes
3,diplomatic_in_scope_records,89,ambassadors; charges_d_affaires,Yes
4,manual_small_official_additions,9,head_of_state; court_of_audit; central_bank_board,Yes
5,manual_category_c_party_boards,79,registered_party_board,Yes
6,manual_workbook_cbb,46,,Yes
7,manual_workbook_crvb,88,,Yes
8,manual_workbook_senior_military,5,,Yes


In [21]:
# -------------------------------------------------------------------
# Combine collection logs and save a run-summary manifest
# -------------------------------------------------------------------

combined_fetch_log_df = pd.concat(
    [
        static_collection_result["fetch_log_dataframe"],
        senate_extraction_result["fetch_log_row"],
        diplomatic_extraction_result["fetch_log_dataframe"]
    ],
    ignore_index=True,
    sort=False
)

combined_fetch_log_output_path = write_csv_safely(
    combined_fetch_log_df,
    FETCH_LOG_PATH
)

# -------------------------------------------------------------------
# Document known cleaning adjustments applied in notebook 02
# -------------------------------------------------------------------

known_cleaning_adjustments_df = pd.DataFrame([
    {
        "adjustment_type": "exclude_out_of_scope_role",
        "person_name": "drs. E. Storm",
        "taxonomy_category": "high_judiciary_crvb",
        "reason": (
            "Listed as a non-judicial CRvB board member. The benchmark "
            "includes only CRvB members with judicial duties."
        ),
        "records_removed": 1
    },
    {
        "adjustment_type": "deduplicate_person_category",
        "person_name": "R.W.L. Koopmans",
        "taxonomy_category": "high_judiciary_crvb",
        "reason": (
            "The same person appears twice in the CRvB source with "
            "different role labels. One person-category record is retained."
        ),
        "records_removed": 1
    }
])

KNOWN_CLEANING_ADJUSTMENTS_PATH = (
    OUTPUT_DIR / "known_benchmark_cleaning_adjustments.csv"
)

write_csv_safely(
    known_cleaning_adjustments_df,
    KNOWN_CLEANING_ADJUSTMENTS_PATH
)

# -------------------------------------------------------------------
# Describe the current live-source collection snapshot.
# -------------------------------------------------------------------
# The diplomatic source is dynamic. The benchmark therefore reflects
# the records returned by the documented collection run rather than a
# historic target count from an earlier development run.

raw_component_record_count = (
    len(curated_v1_df)
    + len(senate_extraction_result["candidate_dataframe"])
    + len(diplomatic_in_scope_df)
    + len(manual_small_additions_df)
    + manual_v2_additions_total
    + len(category_c_manual_df)
)

known_records_removed = int(
    known_cleaning_adjustments_df["records_removed"].sum()
)

expected_final_snapshot_record_count = (
    raw_component_record_count
    - known_records_removed
)

current_diplomatic_category_counts = (
    diplomatic_in_scope_df["taxonomy_category"]
    .value_counts(dropna=False)
    .to_dict()
)

run_summary = {
    "collection_timestamp_utc": COLLECTION_TIMESTAMP_UTC,
    "benchmark_snapshot_type": "current_live_source_snapshot",
    "source_mapping_rows": len(pep_source_mapping_df),
    "generic_static_candidate_records": len(
        static_collection_result["candidate_dataframe"]
    ),
    "static_collection_errors": len(
        static_collection_result["error_log_dataframe"]
    ),
    "senate_candidate_records": len(
        senate_extraction_result["candidate_dataframe"]
    ),
    "senate_profile_full_names_extracted": int(
        senate_extraction_result["candidate_dataframe"][
            "profile_name_extraction_status"
        ].eq("profile_intro_full_name").sum()
    ),
    "diplomatic_all_candidate_records": len(
        diplomatic_all_candidates_df
    ),
    "diplomatic_in_scope_records": len(
        diplomatic_in_scope_df
    ),
    "diplomatic_in_scope_category_counts": current_diplomatic_category_counts,
    "manual_small_additions": len(
        manual_small_additions_df
    ),
    "manual_workbook_additions_raw": manual_v2_additions_total,
    "manual_category_c_records": len(category_c_manual_df),
    "raw_component_records_before_cleaning": raw_component_record_count,
    "documented_cleaning_adjustments": known_records_removed,
    "expected_final_snapshot_records_after_cleaning": (
        expected_final_snapshot_record_count
    ),
    "combined_fetch_log_path": str(
        combined_fetch_log_output_path
    ),
    "component_audit_path": str(
        manual_source_log_output_path
    ),
    "known_cleaning_adjustments_path": str(
        KNOWN_CLEANING_ADJUSTMENTS_PATH
    )
}

with open(
    RUN_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        run_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )

print("Saved collection run summary:")
print(RUN_SUMMARY_PATH)

print("\nKnown cleaning adjustments:")
display(known_cleaning_adjustments_df)

print("\nCollection summary:")
for key, value in run_summary.items():
    print(f"{key}: {value}")

if len(senate_extraction_result["candidate_dataframe"]) != 75:
    raise ValueError(
        "The final Senate component must contain 75 members, but "
        f"contains {len(senate_extraction_result['candidate_dataframe'])}."
    )

if expected_final_snapshot_record_count <= 0:
    raise ValueError(
        "The expected final snapshot size must be positive. "
        "Inspect the component inputs."
    )

print(
    "\nValidation passed: the current collection snapshot contains "
    f"{raw_component_record_count} raw component records. After "
    f"{known_records_removed} documented cleaning adjustments, "
    f"notebook 02 should export {expected_final_snapshot_record_count} "
    "benchmark records."
)


Saved collection run summary:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_run_summary.json

Known cleaning adjustments:


,adjustment_type,person_name,taxonomy_category,reason,records_removed
0,exclude_out_of_scope_role,drs. E. Storm,high_judiciary_crvb,Listed as a non-judicial CRvB board member. Th...,1
1,deduplicate_person_category,R.W.L. Koopmans,high_judiciary_crvb,The same person appears twice in the CRvB sour...,1



Collection summary:
collection_timestamp_utc: 2026-06-22T15:02:25.905779+00:00
source_mapping_rows: 72
generic_static_candidate_records: 328
static_collection_errors: 2
senate_candidate_records: 75
diplomatic_all_candidate_records: 115
diplomatic_in_scope_records: 89
manual_small_additions: 9
manual_workbook_additions_raw: 139
manual_category_c_records: 79
raw_component_records_before_cleaning: 677
known_cleaning_adjustments: 2
expected_final_v3_records_after_cleaning: 675
expected_historic_v3_record_count: 711
combined_fetch_log_path: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\collection_fetch_log.csv
component_audit_path: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\manual_source_components_log.csv
known_cleaning_adjustments_path: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\known_benchmark_cleaning_adjustments.csv

